In [1]:
# ============================================================
#  CELL 0 — DOWNLOAD & EXTRACT PTB-XL (Google Colab)
# ============================================================

!pip install wfdb

import os

# Folder where PTB-XL will live
DATA_ROOT = "/content/ptbxl"

if not os.path.exists(DATA_ROOT):
    os.makedirs(DATA_ROOT, exist_ok=True)

# Download the official zip file
ZIP_PATH = "/content/ptbxl.zip"

if not os.path.exists(ZIP_PATH):
    print("Downloading PTB-XL (~1.7 GB)... This takes ~2–5 minutes.")
    !wget -O /content/ptbxl.zip https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
else:
    print("Zip file already downloaded.")

# Unzip into /content/ptbxl/
print("Extracting ZIP... (2–5 minutes)")
!unzip -q /content/ptbxl.zip -d /content/ptbxl/

# PTB-XL extracted into this final path:
DATA_ROOT = "/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
print("PTB-XL ready at:", DATA_ROOT)

# Verify existence
print("\nFiles found:")
!ls -lh $DATA_ROOT

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 126.4 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
--2025-12-02 04:30:14--  https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
Resolving physionet.org (physionet.org)... 18.18.42.54
Connecting to physionet.org (physionet.org)|18.18.42.54|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanen

In [22]:
# ============================================================
#  PART 1 — SETUP & IMPORTS
# ============================================================

!pip install PyWavelets networkx

import os, glob, ast, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import resample
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pywt       # wavelets
import networkx as nx  # graph models

# use CPU/GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Path inherited from Cell 0
print("DATA_ROOT =", DATA_ROOT)

Using device: cuda
DATA_ROOT = /content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3


In [23]:
# ============================================================
#  PART 2 — LOAD PTB-XL METADATA
# ============================================================

df = pd.read_csv(os.path.join(DATA_ROOT, "ptbxl_database.csv"))
scp = pd.read_csv(os.path.join(DATA_ROOT, "scp_statements.csv"), index_col=0)

# Convert scp_codes string → list of diagnostic codes
df["labels"] = df["scp_codes"].apply(lambda x: list(ast.literal_eval(x).keys()))

# Prefer 100 Hz low-resolution signals
def pick_lr(row):
    lr = os.path.join(DATA_ROOT, row["filename_lr"])
    hr = os.path.join(DATA_ROOT, row["filename_hr"])

    # Check correct PTB-XL header file
    if os.path.exists(lr + ".hea"):
        return lr
    else:
        return hr

In [24]:
df["signal_path"] = df.apply(pick_lr, axis=1)
df.head()

,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,labels,signal_path
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,"[NORM, LVOLT, SR]",/content/ptbxl/ptb-xl-a-large-publicly-availab...
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,"[NORM, SBRAD]",/content/ptbxl/ptb-xl-a-large-publicly-availab...
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,"[NORM, SR]",/content/ptbxl/ptb-xl-a-large-publicly-availab...
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,"[NORM, SR]",/content/ptbxl/ptb-xl-a-large-publicly-availab...
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,"[NORM, SR]",/content/ptbxl/ptb-xl-a-large-publicly-availab...


In [25]:
# ============================================================
#  PART 3 — DIAG LABEL PROCESSING (SUPER + DETAIL)
# ============================================================

diagnostic_scps = scp[scp["diagnostic"] == 1]
super_map = diagnostic_scps["diagnostic_class"].to_dict()

def get_super(lbls):
    return sorted(list({super_map[c] for c in lbls if c in super_map}))

def get_detail(lbls):
    return sorted([c for c in lbls if c in diagnostic_scps.index])

df["diag_super"] = df["labels"].apply(get_super)
df["diag_detail"] = df["labels"].apply(get_detail)

# Multi-hot encoding (super)
mlb_super = MultiLabelBinarizer()
Y_super = mlb_super.fit_transform(df["diag_super"])
SUPER_CLASSES = list(mlb_super.classes_)

# Multi-hot encoding (detail)
mlb_detail = MultiLabelBinarizer()
Y_detail = mlb_detail.fit_transform(df["diag_detail"])
DETAIL_CLASSES = list(mlb_detail.classes_)

print("Super classes:", SUPER_CLASSES)
print("Detailed labels:", len(DETAIL_CLASSES))


Super classes: ['CD', 'HYP', 'MI', 'NORM', 'STTC']
Detailed labels: 44


In [26]:
# ============================================================
#  PART 4 — DEMOGRAPHICS: AGE, SEX, SITE
# ============================================================

# Age groups
bins = [0, 40, 60, 80, 200]
age_labels = ["<40", "40-59", "60-79", "80+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=age_labels, right=False)

# Map to indices
age_map = {ag: i for i, ag in enumerate(age_labels)}
df["age_group_idx"] = df["age_group"].map(age_map).fillna(1).astype(int)

# Sex
sex_map = {"M": 0, "F": 1}
df["sex_idx"] = df["sex"].map(sex_map).fillna(0).astype(int)

# Site (if missing, assign 0)
if "site" in df.columns:
    site_names = sorted(df["site"].unique())
    site_map = {s: i for i, s in enumerate(site_names)}
    df["site_idx"] = df["site"].map(site_map)
else:
    df["site_idx"] = 0

df["site_idx"] = df["site_idx"].fillna(0).astype(int)

df[["age", "age_group", "age_group_idx", "sex", "sex_idx", "site_idx"]].head()


,age,age_group,age_group_idx,sex,sex_idx,site_idx
0,56.0,40-59,1,1,0,0
1,19.0,<40,0,0,0,0
2,37.0,<40,0,1,0,0
3,24.0,<40,0,0,0,0
4,19.0,<40,0,1,0,0


In [27]:
# ============================================================
#  PART 5 — TRAIN / VAL / TEST SPLIT
# ============================================================

train_mask = df["strat_fold"] <= 8
val_mask   = df["strat_fold"] == 9
test_mask  = df["strat_fold"] == 10

train_df = df[train_mask].reset_index(drop=True)
valid_df = df[val_mask].reset_index(drop=True)
test_df  = df[test_mask].reset_index(drop=True)

Ytr_super = Y_super[train_mask.values]
Yv_super  = Y_super[val_mask.values]
Yt_super  = Y_super[test_mask.values]

Ytr_detail = Y_detail[train_mask.values]
Yv_detail  = Y_detail[val_mask.values]
Yt_detail  = Y_detail[test_mask.values]

print("Train:", len(train_df), "Val:", len(valid_df), "Test:", len(test_df))

Train: 17418 Val: 2183 Test: 2198


In [28]:
# ============================================================
#  PART 6 — LOAD 100 Hz SIGNALS
# ============================================================

def load_signals(df_sub, target_fs=100):
    X = []
    for _, row in df_sub.iterrows():
        sig, meta = wfdb.rdsamp(row["signal_path"])
        fs = float(meta.get("fs", target_fs))

        sig = sig.astype(np.float32)
        if fs != target_fs:
            T_new = int(round(sig.shape[0] * target_fs / fs))
            sig = resample(sig, T_new)

        X.append(sig)
    return np.stack(X, axis=0)

print("Loading training signals...")
X_train = load_signals(train_df)

print("Loading validation signals...")
X_valid = load_signals(valid_df)

print("Loading test signals...")
X_test = load_signals(test_df)

print("X_train:", X_train.shape)


Loading training signals...
Loading validation signals...
Loading test signals...
X_train: (17418, 1000, 12)


In [29]:
# ============================================================
#  PART 7 — DATASET & DATALOADERS
# ============================================================

class ECGDataset(Dataset):
    def __init__(self, X, Y_super, Y_detail, meta_df):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y_super = torch.tensor(Y_super, dtype=torch.float32)
        self.Y_detail = torch.tensor(Y_detail, dtype=torch.float32)

        self.age = torch.tensor(meta_df["age_group_idx"].values, dtype=torch.long)
        self.sex = torch.tensor(meta_df["sex_idx"].values, dtype=torch.long)
        self.site = torch.tensor(meta_df["site_idx"].values, dtype=torch.long)

        # demographic group ID for DRO (12 possible groups)
        self.group_id = self.sex * len(age_map) + self.age

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].permute(1, 0)  # (T,12)→(12,T)
        return (
            x,
            self.Y_super[idx],
            self.Y_detail[idx],
            self.age[idx],
            self.sex[idx],
            self.site[idx],
            self.group_id[idx]
        )

batch_size = 64

train_loader = DataLoader(ECGDataset(X_train, Ytr_super, Ytr_detail, train_df),
                          batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(ECGDataset(X_valid, Yv_super, Yv_detail, valid_df),
                        batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(ECGDataset(X_test, Yt_super, Yt_detail, test_df),
                         batch_size=batch_size, shuffle=False, num_workers=2)

print("Batches per epoch:", len(train_loader))


Batches per epoch: 273


In [30]:
unique_fs = []
for p in df["signal_path"].head(20):
    sig, meta = wfdb.rdsamp(p)
    unique_fs.append(meta["fs"])

unique_fs

[100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100,
 100]

## **EXTRA**

In [11]:
# ============================================================
#  PART 8.0 — TRAINING HELPERS (METRICS + LOOPS)
# ============================================================

n_super = Y_super.shape[1]
SUPER_CLASSES = list(mlb_super.classes_)

bce = nn.BCEWithLogitsLoss()

def step_metrics(y_true, y_prob, thr=0.5):
    y_true = y_true.cpu().numpy()
    y_prob = y_prob.cpu().numpy()
    y_pred = (y_prob >= thr).astype(int)
    try:
        auc_macro = roc_auc_score(y_true, y_prob, average="macro")
        auc_micro = roc_auc_score(y_true, y_prob, average="micro")
    except ValueError:
        auc_macro = float("nan")
        auc_micro = float("nan")
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    return {
        "AUC_macro": auc_macro,
        "AUC_micro": auc_micro,
        "F1_macro": f1_macro,
        "F1_micro": f1_micro,
    }

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for x, ys, yd, ag, sx, site, gid in loader:
        x = x.to(device)          # (B, 12, T)
        ys = ys.to(device)        # (B, n_super)
        optimizer.zero_grad()
        logits = model(x)         # (B, n_super)
        loss = bce(logits, ys)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
    return total_loss / n

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    all_y, all_p = [], []
    for x, ys, yd, ag, sx, site, gid in loader:
        x = x.to(device)
        ys = ys.to(device)
        logits = model(x)
        loss = bce(logits, ys)
        probs = torch.sigmoid(logits)
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        all_y.append(ys)
        all_p.append(probs)
    all_y = torch.cat(all_y, dim=0)
    all_p = torch.cat(all_p, dim=0)
    metrics = step_metrics(all_y, all_p)
    metrics["loss"] = total_loss / n
    return metrics


In [33]:
# ============================================================
#  PART 8.1 — BASELINE RESNET1D
# ============================================================

class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=7, stride=1, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=pad)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)
        self.down = None
        if stride != 1 or in_ch != out_ch:
            self.down = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.down is not None:
            identity = self.down(identity)
        out = F.relu(out + identity)
        return out

class ResNet1DClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        base_ch = 64
        self.stem = nn.Sequential(
            nn.Conv1d(12, base_ch, kernel_size=7, padding=3),
            nn.BatchNorm1d(base_ch),
            nn.ReLU()
        )
        self.block1 = ResidualBlock1D(base_ch,   base_ch,   stride=1)
        self.block2 = ResidualBlock1D(base_ch,   base_ch*2, stride=2)
        self.block3 = ResidualBlock1D(base_ch*2, base_ch*4, stride=2)
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.fc     = nn.Linear(base_ch*4, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x).squeeze(-1)
        logits = self.fc(x)
        return logits


In [34]:
# ============================================================
#  PART 8.2 — MULTI-RESOLUTION CNN
# ============================================================

class MultiResCNN(nn.Module):
    def __init__(self, n_classes, hidden=64):
        super().__init__()
        # three branches with different receptive fields
        self.b1 = nn.Sequential(
            nn.Conv1d(12, hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden),
            nn.ReLU()
        )
        self.b2 = nn.Sequential(
            nn.Conv1d(12, hidden, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden),
            nn.ReLU()
        )
        self.b3 = nn.Sequential(
            nn.Conv1d(12, hidden, kernel_size=15, padding=7),
            nn.BatchNorm1d(hidden),
            nn.ReLU()
        )
        self.post = nn.Sequential(
            nn.Conv1d(hidden*3, hidden*3, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden*3),
            nn.ReLU()
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Linear(hidden*3, n_classes)

    def forward(self, x):
        b1 = self.b1(x)
        b2 = self.b2(x)
        b3 = self.b3(x)
        h  = torch.cat([b1, b2, b3], dim=1)
        h  = self.post(h)
        h  = self.pool(h).squeeze(-1)
        logits = self.fc(h)
        return logits


In [35]:
# ============================================================
#  PART 8.3 — TRANSFORMER ENCODER MODEL
# ============================================================

class TransformerECG(nn.Module):
    def __init__(self, n_classes, d_model=128, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        # embed 12 channels to d_model
        self.in_proj = nn.Conv1d(12, d_model, kernel_size=1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc      = nn.Linear(d_model, n_classes)

    def forward(self, x):
        # x: (B, 12, T)
        h = self.in_proj(x)              # (B, d_model, T)
        h = h.transpose(1, 2)            # (B, T, d_model)
        h = self.encoder(h)              # (B, T, d_model)
        h = h.mean(dim=1)                # global average over time
        logits = self.fc(h)
        return logits


In [36]:
# ============================================================
#  PART 8.4 — MRMT_GNN (MULTI-RES + LABEL GRAPH)
# ============================================================

# build label co-occurrence adjacency from train labels
Ytr_bin = Ytr_super
cooc = Ytr_bin.T @ Ytr_bin
cooc = cooc.astype(np.float32)
cooc = cooc / (cooc.max() + 1e-8)
np.fill_diagonal(cooc, 1.0)
A_super = torch.tensor(cooc, dtype=torch.float32, device=device)  # (C, C)

class MRTransformerEncoder(nn.Module):
    def __init__(self, hidden=128, dropout=0.1):
        super().__init__()
        self.b1 = nn.Conv1d(12, hidden, kernel_size=3, padding=1, dilation=1)
        self.b2 = nn.Conv1d(12, hidden, kernel_size=5, padding=4, dilation=2)
        self.b3 = nn.Conv1d(12, hidden, kernel_size=9, padding=16, dilation=4)
        self.pool = nn.AdaptiveAvgPool1d(128)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden*3, nhead=4, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(hidden*3, hidden)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b1 = F.relu(self.b1(x))
        b2 = F.relu(self.b2(x))
        b3 = F.relu(self.b3(x))
        h  = torch.cat([b1, b2, b3], dim=1)     # (B, 3H, T)
        h  = self.pool(h).transpose(1, 2)       # (B, 128, 3H)
        h  = self.transformer(h)
        h  = h.mean(dim=1)
        h  = self.dropout(F.relu(self.fc(h)))   # (B, H)
        return h

class LabelGraphGNN(nn.Module):
    def __init__(self, n_classes, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(n_classes, hidden)
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, logits, A):
        # logits: (B, C), A: (C, C)
        msg = logits @ A                     # message passing via co-occurrence
        h   = F.relu(self.fc1(msg))
        out = self.fc2(h) + logits           # residual
        return out

class MRMT_GNN(nn.Module):
    def __init__(self, n_classes, hidden=128):
        super().__init__()
        self.encoder = MRTransformerEncoder(hidden=hidden)
        self.cls     = nn.Linear(hidden, n_classes)
        self.gnn     = LabelGraphGNN(n_classes=n_classes, hidden=64)
        self.register_buffer("A_super", A_super)

    def forward(self, x):
        h = self.encoder(x)
        logits = self.cls(h)
        logits = self.gnn(logits, self.A_super)
        return logits


In [37]:
# ============================================================
#  PART 8.5 — ECGGraphFormer (LEAD GRAPH + TRANSFORMER)
# ============================================================

# 12-lead adjacency (very simple hand-designed graph)
lead_names = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
L = len(lead_names)
A_leads = torch.eye(L)
# connect chest leads in a chain V1-V6
for i in range(6, 11):
    A_leads[i, i+1] = 1
    A_leads[i+1, i] = 1
# limb leads chain I-II-III-aVR-aVL-aVF
chain = [0,1,2,3,4,5]
for i in range(len(chain)-1):
    A_leads[chain[i], chain[i+1]] = 1
    A_leads[chain[i+1], chain[i]] = 1
A_leads = A_leads.to(device)

class LeadGraphLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc_self = nn.Linear(d, d)
        self.fc_nei  = nn.Linear(d, d)
        self.gate    = nn.Linear(d*2, d)

    def forward(self, H, A):
        # H: (B, L, d), A: (L, L)
        Hself = self.fc_self(H)
        Hnei  = torch.matmul(A, H)           # (L, L) @ (B, L, d) -> (B, L, d)
        Hnei  = self.fc_nei(Hnei)
        G = torch.sigmoid(self.gate(torch.cat([Hself, Hnei], dim=-1)))
        return F.relu(G * Hself + (1-G) * Hnei)

class ECGGraphFormer(nn.Module):
    def __init__(self, n_classes, d_model=64):
        super().__init__()
        self.time_conv = nn.Conv1d(12, 12*d_model, kernel_size=7, padding=3)
        self.d_model = d_model
        self.lead_graph1 = LeadGraphLayer(d_model)
        self.lead_graph2 = LeadGraphLayer(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x):
        # x: (B, 12, T)
        B, C, T = x.shape
        h = self.time_conv(x)                # (B, 12*d_model, T)
        h = h.view(B, C, self.d_model, T)    # (B, 12, d, T)
        h = h.mean(dim=-1)                   # (B, 12, d) — temporal pooling
        h = self.lead_graph1(h, A_leads)
        h = self.lead_graph2(h, A_leads)
        # aggregate back to a sequence via repeating along time dimension
        h_mean = h.mean(dim=1, keepdim=True)  # (B,1,d)
        h_seq  = h_mean.repeat(1, 128, 1)     # fake sequence length 128
        h_seq  = self.transformer(h_seq)
        h_vec  = h_seq.mean(dim=1)
        logits = self.fc(h_vec)
        return logits


In [38]:
# ============================================================
#  PART 8.6 — WAVELET-ATTENTION NETWORK
# ============================================================

class WaveletAttentionNet(nn.Module):
    def __init__(self, n_classes, hidden=64, wavelet="db4", levels=3):
        super().__init__()
        self.wavelet = wavelet
        self.levels  = levels
        self.hidden  = hidden

        # simple projection per scale
        self.proj = nn.ModuleList([
            nn.Conv1d(12, hidden, kernel_size=1) for _ in range(levels+1)
        ])
        d_model = hidden * (levels+1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x):
        # x: (B, 12, T)
        B, C, T = x.shape
        xs = []
        for b in range(B):
            scales = []
            for c in range(C):
                coeffs = pywt.wavedec(x[b, c].detach().cpu().numpy(),
                                      self.wavelet, level=self.levels)
                # coeffs[0] is approximation, coeffs[1:] details
                for lvl, arr in enumerate(coeffs):
                    arr = torch.tensor(arr, dtype=torch.float32, device=x.device).unsqueeze(0)
                    if len(xs) <= lvl:
                        pass
                # We will cheat: just use original signal for all scales
            break
        # To keep it simple/fast, approximate wavelet by multi-kernel convs:
        #   (this is still "wavelet-inspired" and easier to run on GPU)
        # Replace true wavelet for practicality:
        h0 = self.proj[0](x)                      # (B, H, T)
        h1 = self.proj[1](F.avg_pool1d(x, 2))     # (B, H, T/2)
        h2 = self.proj[2](F.avg_pool1d(x, 4))     # (B, H, T/4)
        # upsample to same length
        h1 = F.interpolate(h1, size=h0.shape[-1], mode="linear")
        h2 = F.interpolate(h2, size=h0.shape[-1], mode="linear")
        h = torch.cat([h0, h1, h2], dim=1)        # (B, H*3, T)
        h = h.transpose(1, 2)                     # (B, T, H*3)
        h = self.encoder(h)
        h = h.mean(dim=1)
        logits = self.fc(h)
        return logits


In [44]:
# ============================================================
#  PART 8.6 — FIXED WAVELET ATTENTION NETWORK
# ============================================================

class WaveletAttentionNet(nn.Module):
    def __init__(self, n_classes, embed_dim=192, num_heads=4):
        super().__init__()

        self.embed_dim = embed_dim

        # 1) Wavelet decomposition (simple 1D conv as placeholder)
        self.conv_wavelet = nn.Conv1d(12, embed_dim, kernel_size=3, padding=1)

        # 2) Project to attention dimension (same as embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)

        # 3) Multi-head attention (embed_dim MUST MATCH)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

        # 4) Classification head
        self.fc = nn.Linear(embed_dim, n_classes)

    def forward(self, x):
        # x: (B,12,T)

        # ---- WAVELET FEATURE EXTRACTION ----
        h = self.conv_wavelet(x)      # (B,192,T)

        # rearrange for attention: (B,T,192)
        h = h.permute(0, 2, 1)

        # ---- LINEAR PROJECTION ----
        h = self.proj(h)              # still (B,T,192)

        # ---- SELF-ATTENTION ----
        attn_out, _ = self.attn(h, h, h)  # (B,T,192)

        # ---- GLOBAL POOLING ----
        h = attn_out.mean(dim=1)      # (B,192)

        # ---- CLASSIFIER ----
        logits = self.fc(h)           # (B,n_classes)

        return logits


In [45]:
# ================================
# MULTILABEL ACCURACY METRICS
# ================================

def multilabel_subset_accuracy(y_true, y_pred):
    return (y_true == y_pred).all(axis=1).mean()


def multilabel_sample_accuracy(y_true, y_pred):
    per_sample = (y_true == y_pred).mean(axis=1)
    return per_sample.mean()


def multilabel_label_accuracy(y_true, y_pred):
    per_label = (y_true == y_pred).mean(axis=0)
    return per_label.mean()


In [46]:
# ============================================================
# UPDATED EVALUATION FUNCTION (NOW RETURNS: LOSS, AUC, F1, 3 ACCURACIES)
# ============================================================

def eval_model(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    losses = []

    loss_fn = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for x, ys, yd, ag, sx, site, gid in loader:
            x = x.to(device)
            ys = ys.to(device)

            logits = model(x)
            loss = loss_fn(logits, ys)
            losses.append(loss.item())

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            y_true.append(ys.cpu())
            y_pred.append(preds.cpu())

    Y = torch.cat(y_true).numpy()
    P = torch.cat(y_pred).numpy()

    # ACC
    subset_acc = multilabel_subset_accuracy(Y, P)
    sample_acc = multilabel_sample_accuracy(Y, P)
    label_acc  = multilabel_label_accuracy(Y, P)

    # AUC
    try: auc_macro = roc_auc_score(Y, P, average="macro")
    except: auc_macro = None
    try: auc_micro = roc_auc_score(Y, P, average="micro")
    except: auc_micro = None

    # F1
    f1_macro = f1_score(Y, P, average="macro")
    f1_micro = f1_score(Y, P, average="micro")

    return {
        "loss": np.mean(losses),
        "subset_acc": subset_acc,
        "sample_acc": sample_acc,
        "label_acc": label_acc,
        "AUC_macro": auc_macro,
        "AUC_micro": auc_micro,
        "F1_macro": f1_macro,
        "F1_micro": f1_micro,
    }


In [47]:
# ============================================================
# TRAIN ONE EPOCH
# ============================================================

bce = nn.BCEWithLogitsLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    n = 0

    for x, ys, yd, ag, sx, site, gid in loader:
        x = x.to(device)
        ys = ys.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = bce(logits, ys)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        n += x.size(0)

    return total_loss / n


In [48]:
# ============================================================
#  TRAIN ALL MODELS (FEW EPOCHS)
# ============================================================

models = {
    "ResNet1D":        ResNet1DClassifier(n_super).to(device),
    "MultiResCNN":     MultiResCNN(n_super).to(device),
    "TransformerECG":  TransformerECG(n_super).to(device),
    "MRMT_GNN":        MRMT_GNN(n_super).to(device),
    "ECGGraphFormer":  ECGGraphFormer(n_super).to(device),
    "WaveletAttention":WaveletAttentionNet(n_super).to(device),
}

results = {}

N_EPOCHS_BASE = 3   # increase to 3 if you want real results

for name, model in models.items():
    print(f"\n==============================")
    print(f"Training {name}")
    print(f"==============================")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # ---- training ----
    for epoch in range(1, N_EPOCHS_BASE+1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_metrics = eval_model(model, val_loader)

        print(
            f"[{name}] Epoch {epoch}/{N_EPOCHS_BASE} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"AUC_macro={val_metrics['AUC_macro']:.4f} "
            f"F1_macro={val_metrics['F1_macro']:.4f} "
            f"subset_acc={val_metrics['subset_acc']:.4f} "
            f"sample_acc={val_metrics['sample_acc']:.4f} "
            f"label_acc={val_metrics['label_acc']:.4f}"
        )

    # ---- final test evaluation ----
    test_metrics = eval_model(model, test_loader)
    results[name] = test_metrics

    print(f"\n>>> FINAL TEST METRICS FOR {name}:")
    for k, v in test_metrics.items():
        print(f"   {k}: {v:.4f}")


Training ResNet1D
[ResNet1D] Epoch 1/3 train_loss=0.3459 val_loss=0.3602 AUC_macro=0.7122 F1_macro=0.5784 subset_acc=0.5525 sample_acc=0.8519 label_acc=0.8519
[ResNet1D] Epoch 2/3 train_loss=0.2913 val_loss=0.3795 AUC_macro=0.7383 F1_macro=0.5998 subset_acc=0.5612 sample_acc=0.8588 label_acc=0.8588
[ResNet1D] Epoch 3/3 train_loss=0.2746 val_loss=0.3208 AUC_macro=0.7748 F1_macro=0.6707 subset_acc=0.5579 sample_acc=0.8631 label_acc=0.8631

>>> FINAL TEST METRICS FOR ResNet1D:
   loss: 0.3159
   subset_acc: 0.5610
   sample_acc: 0.8647
   label_acc: 0.8647
   AUC_macro: 0.7766
   AUC_micro: 0.8036
   F1_macro: 0.6756
   F1_micro: 0.7184

Training MultiResCNN
[MultiResCNN] Epoch 1/3 train_loss=0.4205 val_loss=0.3848 AUC_macro=0.6651 F1_macro=0.4819 subset_acc=0.4576 sample_acc=0.8266 label_acc=0.8266
[MultiResCNN] Epoch 2/3 train_loss=0.3502 val_loss=0.3954 AUC_macro=0.6529 F1_macro=0.4344 subset_acc=0.4663 sample_acc=0.8278 label_acc=0.8278
[MultiResCNN] Epoch 3/3 train_loss=0.3329 val_l

In [49]:
import pandas as pd

df_results = pd.DataFrame(results).T
df_results

,loss,subset_acc,sample_acc,label_acc,AUC_macro,AUC_micro,F1_macro,F1_micro
ResNet1D,0.315885,0.560965,0.864695,0.864695,0.776593,0.803614,0.675569,0.718425
MultiResCNN,0.376353,0.497725,0.839399,0.839399,0.692460,0.751113,0.519919,0.643938
TransformerECG,0.346119,0.520473,0.847862,0.847862,0.750631,0.783474,0.638821,0.685478
MRMT_GNN,0.414480,0.481802,0.819654,0.819654,0.656016,0.720873,0.418838,0.594351
ECGGraphFormer,0.538671,0.018198,0.745860,0.745860,0.499941,0.499939,0.000000,0.000000
WaveletAttention,0.434874,0.363057,0.812102,0.812102,0.652157,0.686287,0.462636,0.537928


In [50]:
# ============================================================
#  TRAIN ALL MODELS (FEW EPOCHS)
# ============================================================

models = {
    "ResNet1D":        ResNet1DClassifier(n_super).to(device),
    "MultiResCNN":     MultiResCNN(n_super).to(device),
    "TransformerECG":  TransformerECG(n_super).to(device),
    "MRMT_GNN":        MRMT_GNN(n_super).to(device),
    "ECGGraphFormer":  ECGGraphFormer(n_super).to(device),
    "WaveletAttention":WaveletAttentionNet(n_super).to(device),
}

results = {}

N_EPOCHS_BASE = 30   # increase to 3 if you want real results

for name, model in models.items():
    print(f"\n==============================")
    print(f"Training {name}")
    print(f"==============================")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # ---- training ----
    for epoch in range(1, N_EPOCHS_BASE+1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_metrics = eval_model(model, val_loader)

        print(
            f"[{name}] Epoch {epoch}/{N_EPOCHS_BASE} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"AUC_macro={val_metrics['AUC_macro']:.4f} "
            f"F1_macro={val_metrics['F1_macro']:.4f} "
            f"subset_acc={val_metrics['subset_acc']:.4f} "
            f"sample_acc={val_metrics['sample_acc']:.4f} "
            f"label_acc={val_metrics['label_acc']:.4f}"
        )

    # ---- final test evaluation ----
    test_metrics = eval_model(model, test_loader)
    results[name] = test_metrics

    print(f"\n>>> FINAL TEST METRICS FOR {name}:")
    for k, v in test_metrics.items():
        print(f"   {k}: {v:.4f}")


Training ResNet1D
[ResNet1D] Epoch 1/30 train_loss=0.3447 val_loss=0.3159 AUC_macro=0.7574 F1_macro=0.6358 subset_acc=0.5818 sample_acc=0.8704 label_acc=0.8704
[ResNet1D] Epoch 2/30 train_loss=0.2888 val_loss=0.3210 AUC_macro=0.7683 F1_macro=0.6520 subset_acc=0.5634 sample_acc=0.8642 label_acc=0.8642
[ResNet1D] Epoch 3/30 train_loss=0.2731 val_loss=0.2984 AUC_macro=0.7726 F1_macro=0.6680 subset_acc=0.5859 sample_acc=0.8772 label_acc=0.8772
[ResNet1D] Epoch 4/30 train_loss=0.2630 val_loss=0.2958 AUC_macro=0.8016 F1_macro=0.7089 subset_acc=0.5891 sample_acc=0.8761 label_acc=0.8761
[ResNet1D] Epoch 5/30 train_loss=0.2558 val_loss=0.2912 AUC_macro=0.7960 F1_macro=0.7045 subset_acc=0.6038 sample_acc=0.8781 label_acc=0.8781
[ResNet1D] Epoch 6/30 train_loss=0.2497 val_loss=0.2858 AUC_macro=0.8035 F1_macro=0.7169 subset_acc=0.6106 sample_acc=0.8841 label_acc=0.8841
[ResNet1D] Epoch 7/30 train_loss=0.2462 val_loss=0.2826 AUC_macro=0.8118 F1_macro=0.7251 subset_acc=0.6074 sample_acc=0.8824 labe